# short_clean_v3 후처리 실험 노트북

현재 최고 제출본 `submit_12_top1_absShort_clean_v3.csv`를 다시 생성하지 않고, **가벼운 후처리/정규화만 적용**해서 새 제출본 3개를 만듭니다.

생성 파일:
- `submit_17_post_clean_norm_v1.csv`
- `submit_18_post_clean_firstsent_v1.csv`
- `submit_19_post_clean_selective_v1.csv`

전략:
1. 기본 정규화 + 반복 표현 제거
2. 첫 문장만 유지하는 보수적 트림
3. 너무 긴/지저분한 케이스만 선택적으로 줄이는 하이브리드


In [ ]:
import sys, pandas as pd
print('python', sys.version)
print('pandas', pd.__version__)


In [ ]:
# -*- coding: utf-8 -*-
from pathlib import Path
import re
import pandas as pd

# ============================================================
# CONFIG
# ============================================================
BEST_SUBMISSION_PATH = Path(
    "/root/upstage-nlp-nlp/code/prediction/qwen3_response_only_best_strategy_8b/manual_submissions/submit_12_top1_absShort_clean_v3.csv"
)
OUT_DIR = BEST_SUBMISSION_PATH.parent

# ============================================================
# 후처리 함수
# ============================================================
def normalize_basic(text: str) -> str:
    text = str(text)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    text = re.sub(r"<\|.*?\|>", "", text)
    text = re.sub(r"<[^>]+>", "", text)
    text = text.replace("/no_think", " ")
    text = re.sub(r"^요약\s*:\s*", "", text).strip()
    text = re.sub(r"#\s*Person\s*(\d+)\s*#", r"#Person\1#", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"([,.!?])\1+", r"\1", text)
    return text


def collapse_exact_repeat(text: str) -> str:
    text = text.strip()
    n = len(text)
    for k in range(8, n // 2 + 1):
        left = text[:k].strip()
        right = text[k:].strip()
        if left and left == right:
            return left
    return text


def collapse_repeated_phrase(text: str) -> str:
    text = re.sub(r"\b(.+?)\s+\1\b", r"\1", text)
    text = re.sub(r"(#Person\d+#(?:은|는|이|가|과|와|에게|을|를)?)\s+\1", r"\1", text)
    return text


def first_sentence_only(text: str) -> str:
    text = text.strip()
    if not text:
        return text
    parts = re.split(r"(?<=[.!?])\s+", text)
    if parts:
        return parts[0].strip()
    return text


def trim_after_second_clause(text: str) -> str:
    text = text.strip()
    if len(text) <= 75:
        return text
    clauses = re.split(r"[,;]\s*", text)
    if len(clauses) >= 3:
        candidate = ", ".join(clauses[:2]).strip()
        if len(candidate) >= 20:
            return candidate
    return text


def post_norm_v1(text: str) -> str:
    text = normalize_basic(text)
    text = collapse_exact_repeat(text)
    text = collapse_repeated_phrase(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def post_firstsent_v1(text: str) -> str:
    text = post_norm_v1(text)
    text = first_sentence_only(text)
    return text.strip()


def post_selective_v1(text: str) -> str:
    base = post_norm_v1(text)
    first = first_sentence_only(base)
    trimmed = trim_after_second_clause(first)

    # 너무 길거나 문장부호가 여러 개면 더 짧은 후보 선택
    punctuation_cnt = len(re.findall(r"[.!?]", base))
    if punctuation_cnt >= 2:
        return trimmed
    if len(base) >= 80:
        return trimmed
    if base.count(",") >= 2 and len(trimmed) >= 20:
        return trimmed
    return base


def save_submission(df: pd.DataFrame, summaries, out_path: Path):
    out = pd.DataFrame({"fname": df["fname"], "summary": summaries})
    out.to_csv(out_path, index=False)
    print("saved:", out_path)


# ============================================================
# 실행
# ============================================================
if not BEST_SUBMISSION_PATH.exists():
    raise FileNotFoundError(f"best submission not found: {BEST_SUBMISSION_PATH}")

base_df = pd.read_csv(BEST_SUBMISSION_PATH)
if "fname" not in base_df.columns or "summary" not in base_df.columns:
    raise ValueError("input csv must have fname, summary columns")

norm_v1 = [post_norm_v1(x) for x in base_df["summary"].tolist()]
firstsent_v1 = [post_firstsent_v1(x) for x in base_df["summary"].tolist()]
selective_v1 = [post_selective_v1(x) for x in base_df["summary"].tolist()]

save_submission(base_df, norm_v1, OUT_DIR / "submit_17_post_clean_norm_v1.csv")
save_submission(base_df, firstsent_v1, OUT_DIR / "submit_18_post_clean_firstsent_v1.csv")
save_submission(base_df, selective_v1, OUT_DIR / "submit_19_post_clean_selective_v1.csv")

preview = pd.DataFrame({
    "original": base_df["summary"].head(10),
    "norm_v1": norm_v1[:10],
    "firstsent_v1": firstsent_v1[:10],
    "selective_v1": selective_v1[:10],
})
print("\n===== preview =====")
print(preview)
